# Сборка аналитического датасета и EDA: ФП → Дефолт

**Контрольная точка:** КТ-3 (Результаты разведочного анализа)  
**Этап:** Шаг 3. Разведочный анализ и визуализация

---

### Цель ноутбука

Собрать единый аналитический датасет из трёх источников и провести разведочный анализ (EDA), чтобы ответить на вопрос: **какие факторы проблемности (ФП) связаны с дефолтом компании?**

### Единица наблюдения

**Компания (ИНН)**. Каждая строка итогового датасета — одна уникальная компания.

### Источники данных

| Источник | Роль | Таблица в Озере | Локальный файл | Строк | Колонок | Период |
|----------|------|----------------|----------------|------:|--------:|--------|
| **CRM** | **Основной** (ФП + вселенная) | `sandbox_ai.tmp_ecp_crm_fp_svy` | `data_crm.csv` | 1 504 263 | 41 | 2022-04 — 2025-12 |
| **H2.0** | Вспомогательный (сегмент) | `sandbox_ai.tmp_h20_fp_svy` | `data_h20.xlsx` | 172 981 | 11 | 2023-02 — 2025-12 |
| **Дефолты** | **Основной** (целевая переменная) | `sandbox_ai.tmp_defaults_svy` | `default_data.pkl` | — | — | 2022 — 2025 |

### Ключевые решения по архитектуре

1. **ФП берём из CRM** — колонка `NUMBER_FP_SFP` (тип ФП, 100% заполненность). Дата выявления — `IDENTIFICATION_DATE`. Источники: Н2.0, Справочник CRM-системы, CRM-система.
2. **Реестр компаний** берём из CRM (`X_INN`). Компании без ФП в датасет не попадают.
3. **Временной фильтр**: ФП считаются только если выявлены **до** дефолта (или до отсечки для недефолтных) — исключает data leakage.
4. **Все компании** в выгрузке дефолтов получают `default_flag = 1`, тяжесть классифицируется отдельно.

### Структура итогового датасета

| Колонка | Тип | Описание |
|---------|-----|----------|
| `inn` | str | ИНН компании (ключ) |
| `default_flag` | int8 | 1 = дефолт, 0 = нет |
| `default_date` | datetime | Дата первого дефолта (NaT для недефолтных) |
| `severity` | str | Тяжесть: тяжёлый / активный / надзор / вышел из дефолта / нет дефолта |
| `fp_*` | int8 | N бинарных колонок по числу уникальных NUMBER_FP_SFP: 1 = ФП был, 0 = нет |

### Разделы EDA (deliverables КТ-3)

| № | Раздел | Что выдаёт |
|---|--------|------------|
| 7 | Гистограммы ФП | Распределение числа ФП у дефолтных vs недефолтных |
| 8 | Динамика дефолтов | По месяцам и кварталам |
| 9.1 | Тепловые карты | Корреляции между ТОП-30 ФП по сегментам |
| 10.2 | Кросс-таблицы | Частота, доля дефолтов, Lift по каждому ФП по сегментам |
| 10.3 | Комбинации | ТОП-5 пар и троек ФП по Lift по сегментам |


## 0. Импорты и настройки

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from itertools import combinations

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

plt.rcParams.update({"font.family": "sans-serif",
                      "font.sans-serif": ["Arial", "DejaVu Sans"],
                      "axes.unicode_minus": False})


## 1. Загрузка данных

In [ ]:
DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"

SEGMENT_MAP = {
    "ДМСБ (микро)":   "МкБ",
    "ДМСБ (малый)":   "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ":            "ККБ",
}
SEG_ORDER = ["МкБ", "МСБ", "ККБ"]

ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

def normalize_inn(s):
    """Приводит ИНН к единому строковому формату (10 или 12 знаков).
    Убирает '.0' от Excel, ведущие нули, дополняет до стандартной длины."""
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)

# --- CRM: основной источник ФП ---
df_crm = pd.read_csv(f"{DATA_DIR}/crm_last_version.csv", dtype=str, low_memory=False)
df_crm.columns = df_crm.columns.str.strip()
df_crm = df_crm.rename(columns={
    'VAL.1': 'VAL_1', 'DESC_TEXT.1': 'DESC_TEXT_1', 'ROW_ID.1': 'ROW_ID_1',
    'AGREEMENT_NUM.1': 'AGREEMENT_NUM_1', 'ROW_ID.2': 'ROW_ID_2',
    'VAL.2': 'VAL_2', 'NAME.1': 'NAME_1', 'VAL.3': 'VAL_3', 'VAL.5': 'VAL_5',
})

# --- H2O: вспомогательный источник ---
df_h2o = pd.read_excel(f"{DATA_DIR}/2023-2025_h20_data.xlsx", dtype=str)
df_h2o.columns = df_h2o.columns.str.strip()
df_h2o = df_h2o.rename(columns={
    'ИНН': 'inn', 'Наименование клиента': 'client_name',
    'CDI ID клиента': 'cdi_id', 'ОПФ': 'legal_form', 'РФ': 'rf',
    'Сегмент (зона ответственности)': 'segment', 'Вид мониторинга': 'mon_type',
    'ИД фактора на клиенте в CRM': 'crm_fp_id',
    'ИД справочного фактора в CRM': 'h20_fp_id',
    'Номер фактора': 'ref_book_fp_id',
    'Дата выявления фактора проблемности': 'fp_start_date',
})
df_h2o["inn"] = df_h2o["inn"].apply(normalize_inn)

# --- Дефолты: история дефолтов компаний ---
df_def = pd.read_pickle(f"{DATA_DIR}/default_data.pkl")
df_def = df_def.astype(str).replace("nan", np.nan)
df_def.columns = df_def.columns.str.strip()
df_def = df_def.rename(columns={'start': 'start_date', 'cure': 'cure_date', 'finish': 'finish_date'})
df_def["inn"] = df_def["inn"].apply(normalize_inn)


## 2. Подготовка таблицы дефолтов

Все компании в выгрузке defaults подверглись дефолту. Тяжесть:
- **unlimited_default=1** → тяжёлый дефолт (бессрочный, присваивается сразу, без дальнейших проверок)
- **writeoff или liquidation заполнены** → тяжёлый дефолт (списание или ликвидация — это даты, заполняется одно из двух)
- **finish_date заполнена** → компания вышла из дефолта (излечилась)
- **cure_date заполнена, finish_date пуста** → компания на стадии надзора (6 мес)
- Остальные → активный дефолт

`default_flag = 1` для **всех** компаний из этой выгрузки.

In [ ]:
# Приводим колонки дат к datetime
for col in ["start_date", "cure_date", "finish_date", "writeoff", "liquidation"]:
    if col in df_def.columns:
        df_def[col] = pd.to_datetime(df_def[col], dayfirst=True, errors="coerce")

# unlimited_default — единственный флаг 0/1
if "unlimited_default" in df_def.columns:
    df_def["unlimited_default"] = pd.to_numeric(df_def["unlimited_default"], errors="coerce").fillna(0).astype(int)

def classify_severity(row):
    """Классифицирует тяжесть дефолта по приоритету (от худшего к лучшему):
    
    1. «тяжёлый»          — unlimited_default=1 (бессрочный, сразу тяжелейший)
                            или writeoff/liquidation заполнены (списание/ликвидация)
    2. «активный»         — компания в процессе дефолта (нет cure_date, нет finish_date)
    3. «надзор»           — есть cure_date, но finish_date ещё не наступила
    4. «вышел из дефолта» — finish_date заполнена (дефолт закрыт)
    """
    if row.get("unlimited_default") == 1:
        return "тяжёлый"
    if pd.notna(row.get("writeoff")) or pd.notna(row.get("liquidation")):
        return "тяжёлый"
    if pd.notna(row.get("finish_date")):
        return "вышел из дефолта"
    if pd.notna(row.get("cure_date")):
        return "надзор"
    return "активный"

df_def["severity"] = df_def.apply(classify_severity, axis=1)



In [ ]:
# Агрегация до уровня компании: одна строка на ИНН.
# У одной компании может быть несколько записей о дефолте (повторные эпизоды).
# Логика: берём наихудшую тяжесть и самую раннюю дату начала дефолта.

# Приоритет тяжести: 0 = худший (тяжёлый), 3 = лучший (вышел)
severity_priority = {"тяжёлый": 0, "активный": 1, "надзор": 2, "вышел из дефолта": 3}
df_def["_sev_rank"] = df_def["severity"].map(severity_priority)

# Фильтруем по тому же периоду, что и CRM/H2O
date_from = pd.Timestamp("2023-01-01")
date_to   = pd.Timestamp("2025-12-31")

defaults = (
    df_def
    .dropna(subset=["inn", "start_date"])
    .query("@date_from <= start_date <= @date_to")
    .sort_values("_sev_rank")
    .groupby("inn", as_index=False)
    .agg(
        default_date=("start_date", "min"),      # самая ранняя дата дефолта
        severity=("severity", "first"),           # наихудшая тяжесть (first после sort)
        writeoff=("writeoff", "min"),             # самая ранняя дата списания (NaT если не было)
        liquidation=("liquidation", "min"),       # самая ранняя дата ликвидации (NaT если не было)
        finish_date=("finish_date", "min"),       # самая ранняя дата закрытия дефолта
        cure_date=("cure_date", "min"),           # самая ранняя дата выхода на надзор
        unlimited_default=("unlimited_default", "max"),  # 1, если есть бессрочный
    )
)
defaults["default_flag"] = 1  # все компании из этой таблицы — дефолтные



## 3. Подготовка таблицы ФП из CRM

`NUMBER_FP_SFP` — тип ФП (100% заполненность). `IDENTIFICATION_DATE` — дата выявления ФП.

CRM — мастер-система, содержит все ФП. Фильтруем по источникам (Н2.0, Справочник CRM-системы, CRM-система) и по периоду 01.01.2023 — 31.12.2025.

In [ ]:
# Конвертируем дату выявления ФП в datetime
df_crm["IDENTIFICATION_DATE"] = pd.to_datetime(
    df_crm["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce"
)

# Нормализация ИНН (единый подход для всех тетрадок)
df_crm["X_INN"] = df_crm["X_INN"].apply(normalize_inn)

# Фильтруем по источникам
df_crm = df_crm[df_crm["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()

# Маппинг сегментов и исключение немаппированных (единый подход)
df_crm["X_AREA_RESP"] = df_crm["X_AREA_RESP"].astype(str).str.strip()
df_crm["segment"] = df_crm["X_AREA_RESP"].map(SEGMENT_MAP)
unmapped = df_crm[df_crm["segment"].isna()]["X_AREA_RESP"].value_counts()
if len(unmapped) > 0:
df_crm = df_crm[df_crm["segment"].notna()].copy()

# Фильтруем по периоду (единый для всех отчётов)
date_from = pd.Timestamp("2023-01-01")
date_to   = pd.Timestamp("2025-12-31")
df_crm = df_crm[
    (df_crm["IDENTIFICATION_DATE"] >= date_from) &
    (df_crm["IDENTIFICATION_DATE"] <= date_to)
].copy()

# Формируем таблицу ФП из CRM
fp = df_crm[["X_INN", "NUMBER_FP_SFP", "IDENTIFICATION_DATE", "segment"]].copy()
fp = fp.rename(columns={
    "X_INN": "inn",
    "NUMBER_FP_SFP": "fp_type",
    "IDENTIFICATION_DATE": "fp_date",
})

fp = fp.dropna(subset=["inn", "fp_type"])

# --- Сегмент компании (мода по всем записям в CRM) ---
_seg = df_crm[["X_INN", "segment"]].dropna(subset=["X_INN"]).drop_duplicates()
_mode = lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else None
seg_company = (
    _seg.groupby("X_INN")
    .agg(segment=("segment", _mode))
    .reset_index()
    .rename(columns={"X_INN": "inn"})
)



## 4. Реестр компаний

Реестр берём из **CRM** (мастер-система). Присоединяем флаг и тяжесть дефолта.

In [ ]:
# Уникальные ИНН из CRM (мастер-система)
companies = fp[["inn"]].drop_duplicates().copy()

# LEFT JOIN с дефолтами
companies = companies.merge(
    defaults[["inn", "default_flag", "default_date", "severity"]],
    on="inn", how="left"
)
companies["default_flag"] = companies["default_flag"].fillna(0).astype(int)
companies["severity"] = companies["severity"].fillna("нет дефолта")

# LEFT JOIN с сегментами
companies = companies.merge(seg_company[["inn", "segment"]], on="inn", how="left")

for seg in SEG_ORDER:
    sub = companies[companies["segment"] == seg]
    if len(sub) > 0:


## 5. Временной фильтр ФП

**Критично для корректности:**
- Для дефолтных: оставляем ФП, выявленные **до** даты дефолта
- Для недефолтных: оставляем ФП, выявленные **до** даты отсечки

In [ ]:
# Дата отсечки для недефолтных компаний: конец периода наблюдения.
# Для них учитываем все ФП, выявленные до этой даты.
CUTOFF = pd.Timestamp("2025-12-31")

# Присоединяем к каждому ФП информацию о дефолте компании
fp_with_def = fp.merge(
    companies[["inn", "default_flag", "default_date"]], on="inn", how="inner"
)

# Референсная дата: дефолтные → дата дефолта, недефолтные → CUTOFF
fp_with_def["ref_date"] = fp_with_def["default_date"].fillna(CUTOFF)

# ГЛАВНЫЙ ФИЛЬТР: оставляем только ФП, выявленные СТРОГО ДО референсной даты.
# Это исключает data leakage — ФП, появившиеся после дефолта, не могли его «предсказать».
before = len(fp_with_def)
fp_filtered = fp_with_def[fp_with_def["fp_date"] < fp_with_def["ref_date"]].copy()
after = len(fp_filtered)



## 6. Pivot — широкий формат

In [ ]:
# Вспомогательная колонка для pivot: наличие ФП = 1
fp_filtered["value"] = 1

# Pivot: длинная таблица (1 строка = 1 ФП) → широкая (1 строка = 1 компания).
# Колонки = типы ФП (NUMBER_FP_SFP), значения = 0/1.
# aggfunc="max": если один тип ФП встретился несколько раз — всё равно 1.
fp_wide = (
    fp_filtered
    .pivot_table(index="inn", columns="fp_type", values="value",
                 aggfunc="max", fill_value=0)
    .reset_index()
)
fp_wide.columns.name = None

# Собираем итоговый датасет: fp_wide + информация о дефолте из universe.
# INNER JOIN: в датасет попадают только компании, у которых есть хотя бы один ФП.
meta_cols = ["inn", "default_flag", "default_date", "severity", "segment"]
dataset = fp_wide.merge(companies[meta_cols], on="inn", how="left")
dataset["default_flag"] = dataset["default_flag"].fillna(0).astype("int8")
dataset["severity"] = dataset["severity"].fillna("нет дефолта")

# Определяем колонки ФП (всё, что не мета-данные)
fp_cols = [c for c in dataset.columns if c not in meta_cols]
dataset[fp_cols] = dataset[fp_cols].fillna(0).astype("int8")

n_def_in_companies = int(companies["default_flag"].sum())
n_def_in_dataset = int(dataset["default_flag"].sum())
n_lost = n_def_in_companies - n_def_in_dataset

if n_lost > 0:

for seg in SEG_ORDER:
    sub = dataset[dataset["segment"] == seg]
    n_d = int(sub["default_flag"].sum())
no_seg = dataset["segment"].isna().sum()
if no_seg > 0:


## 7. Гистограммы распределения ФП: дефолтные vs недефолтные

In [ ]:
# Считаем общее число ФП на каждую компанию (сумма по бинарным колонкам)
fp_per_company = dataset[fp_cols].sum(axis=1)
dataset["fp_count"] = fp_per_company

def_counts = fp_per_company[dataset["default_flag"] == 1]
nodef_counts = fp_per_company[dataset["default_flag"] == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Обрезаем по 95-му перцентилю, чтобы выбросы не сжимали основную часть гистограммы
max_fp = int(fp_per_company.quantile(0.95)) + 1
bins = range(0, max_fp + 2)

# Левый график: абсолютные числа (показывает масштаб дисбаланса классов)
axes[0].hist(nodef_counts.clip(upper=max_fp), bins=bins, alpha=0.7, label="Недефолтные", color="steelblue", edgecolor="white")
axes[0].hist(def_counts.clip(upper=max_fp), bins=bins, alpha=0.7, label="Дефолтные", color="tomato", edgecolor="white")
axes[0].set_xlabel("Количество ФП")
axes[0].set_ylabel("Число компаний")
axes[0].set_title("Распределение количества ФП")
axes[0].legend()

# Правый график: плотность (нормализованные, чтобы сравнивать форму распределений)
axes[1].hist(nodef_counts.clip(upper=max_fp), bins=bins, alpha=0.7, density=True, label="Недефолтные", color="steelblue", edgecolor="white")
axes[1].hist(def_counts.clip(upper=max_fp), bins=bins, alpha=0.7, density=True, label="Дефолтные", color="tomato", edgecolor="white")
axes[1].set_xlabel("Количество ФП")
axes[1].set_ylabel("Доля")
axes[1].set_title("Нормализованное распределение (плотность)")
axes[1].legend()

plt.tight_layout()
plt.show()

### 7.1. Гистограммы распределения ФП по сегментам

In [ ]:
seg_stats = []

for seg in SEG_ORDER:
    seg_data = dataset[dataset["segment"] == seg]
    if len(seg_data) == 0:
        continue
    fp_per = seg_data[fp_cols].sum(axis=1)
    d = fp_per[seg_data["default_flag"] == 1]
    nd = fp_per[seg_data["default_flag"] == 0]

    seg_stats.append({
        "Сегмент": seg,
        "Компаний": len(seg_data),
        "Дефолтных": int(seg_data["default_flag"].sum()),
        "DR": f"{seg_data['default_flag'].mean():.2%}",
        "Среднее ФП (деф.)": round(d.mean(), 1) if len(d) > 0 else "-",
        "Среднее ФП (недеф.)": round(nd.mean(), 1) if len(nd) > 0 else "-",
        "Медиана ФП (деф.)": int(d.median()) if len(d) > 0 else "-",
        "Медиана ФП (недеф.)": int(nd.median()) if len(nd) > 0 else "-",
        "Макс ФП (деф.)": int(d.max()) if len(d) > 0 else "-",
        "Макс ФП (недеф.)": int(nd.max()) if len(nd) > 0 else "-",
    })

    max_fp_seg = int(fp_per.quantile(0.95)) + 1
    bins_seg = range(0, max_fp_seg + 2)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(nd.clip(upper=max_fp_seg), bins=bins_seg, alpha=0.7, label="Недефолтные", color="steelblue", edgecolor="white")
    axes[0].hist(d.clip(upper=max_fp_seg), bins=bins_seg, alpha=0.7, label="Дефолтные", color="tomato", edgecolor="white")
    axes[0].set_xlabel("Количество ФП")
    axes[0].set_ylabel("Число компаний")
    axes[0].set_title(f"{seg} — распределение количества ФП")
    axes[0].legend()

    axes[1].hist(nd.clip(upper=max_fp_seg), bins=bins_seg, alpha=0.7, density=True, label="Недефолтные", color="steelblue", edgecolor="white")
    axes[1].hist(d.clip(upper=max_fp_seg), bins=bins_seg, alpha=0.7, density=True, label="Дефолтные", color="tomato", edgecolor="white")
    axes[1].set_xlabel("Количество ФП")
    axes[1].set_ylabel("Доля")
    axes[1].set_title(f"{seg} — нормализованное распределение")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

display(pd.DataFrame(seg_stats).style.hide(axis="index"))

## 8. Закрытие дефолтов по месяцам / кварталам

Дата окончания дефолта (`end_date`) определяется по приоритету: `writeoff` → `liquidation` → `finish_date` → `cure_date`.
Если ни одна дата не заполнена — дефолт считается открытым.

Графики показывают количество компаний, у которых `end_date` попадает в 2023–2025, **без привязки к `start_date`**.

In [ ]:
# --- Привязка сегмента к дефолтам ---
df_def = df_def.merge(seg_company[["inn", "segment"]], on="inn", how="left")
defaults = defaults.merge(seg_company[["inn", "segment"]], on="inn", how="left")

# --- Единая дата окончания дефолта (приоритет: writeoff → liquidation → finish_date → cure_date) ---
df_def["end_date"] = (
    df_def["writeoff"]
    .combine_first(df_def["liquidation"])
    .combine_first(df_def["finish_date"])
    .combine_first(df_def["cure_date"])
)
df_def["end_date"] = pd.to_datetime(df_def["end_date"])

defaults["end_date"] = (
    defaults["writeoff"]
    .combine_first(defaults["liquidation"])
    .combine_first(defaults["finish_date"])
    .combine_first(defaults["cure_date"])
)
defaults["end_date"] = pd.to_datetime(defaults["end_date"])

closed_all = df_def[df_def["end_date"].notna()].copy()
closed_all = closed_all[
    (closed_all["end_date"] >= pd.Timestamp("2023-01-01")) &
    (closed_all["end_date"] <= pd.Timestamp("2025-12-31"))
]
closed_all["end_month"]   = closed_all["end_date"].dt.to_period("M")
closed_all["end_quarter"] = closed_all["end_date"].dt.to_period("Q")

monthly  = closed_all.groupby("end_month")["inn"].nunique()
monthly.index = monthly.index.astype(str)
quarterly = closed_all.groupby("end_quarter")["inn"].nunique()
quarterly.index = quarterly.index.astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].bar(range(len(monthly)), monthly.values, color="indianred", edgecolor="white")
axes[0].set_xticks(range(len(monthly)))
axes[0].set_xticklabels(monthly.index, rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Количество компаний")
axes[0].set_title("Закрытие дефолтов по месяцам (за период 2023-2025, с любой датой начала дефолта)")

axes[1].bar(range(len(quarterly)), quarterly.values, color="coral", edgecolor="white")
axes[1].set_xticks(range(len(quarterly)))
axes[1].set_xticklabels(quarterly.index, rotation=45, ha="right")
axes[1].set_ylabel("Количество компаний")
axes[1].set_title("Закрытие дефолтов по кварталам (за период 2023-2025, с любой датой начала дефолта)")

plt.tight_layout()
plt.show()



In [ ]:
# --- Статистика: закрыт / открыт ---
closed = df_def["end_date"].notna().sum()
opened = df_def["end_date"].isna().sum()

closed_p = defaults["end_date"].notna().sum()
opened_p = defaults["end_date"].isna().sum()

# --- Проверка: ИНН с несколькими различными start_date ---
multi_start = (
    df_def.groupby("inn")["start_date"]
    .nunique()
    .loc[lambda x: x > 1]
    .sort_values(ascending=False)
)
if len(multi_start) > 0:
    for inn in multi_start.head(5).index:
        dates = sorted(df_def.loc[df_def["inn"] == inn, "start_date"].dropna().unique())


### 8.1. Закрытие дефолтов по месяцам / кварталам (start_date и end_date в периоде)

Те же закрытые дефолты, но дополнительно фильтруем: и `start_date`, и `end_date` должны попадать в 2023–2025.
Это позволяет увидеть дефолты, которые **целиком** прошли внутри анализируемого периода.

In [ ]:
# end_date уже рассчитана выше; фильтруем: И start_date, И end_date в 2023–2025
closed_both = df_def[
    df_def["end_date"].notna() &
    (df_def["start_date"] >= pd.Timestamp("2023-01-01")) &
    (df_def["start_date"] <= pd.Timestamp("2025-12-31")) &
    (df_def["end_date"]   >= pd.Timestamp("2023-01-01")) &
    (df_def["end_date"]   <= pd.Timestamp("2025-12-31"))
].copy()
closed_both["end_month"]   = closed_both["end_date"].dt.to_period("M")
closed_both["end_quarter"] = closed_both["end_date"].dt.to_period("Q")

monthly_b  = closed_both.groupby("end_month")["inn"].nunique()
monthly_b.index = monthly_b.index.astype(str)
quarterly_b = closed_both.groupby("end_quarter")["inn"].nunique()
quarterly_b.index = quarterly_b.index.astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].bar(range(len(monthly_b)), monthly_b.values, color="indianred", edgecolor="white")
axes[0].set_xticks(range(len(monthly_b)))
axes[0].set_xticklabels(monthly_b.index, rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Количество компаний")
axes[0].set_title("Закрытие дефолтов по месяцам (дата начала дефолта находится в диапазоне 2023-2025)")

axes[1].bar(range(len(quarterly_b)), quarterly_b.values, color="coral", edgecolor="white")
axes[1].set_xticks(range(len(quarterly_b)))
axes[1].set_xticklabels(quarterly_b.index, rotation=45, ha="right")
axes[1].set_ylabel("Количество компаний")
axes[1].set_title("Закрытие дефолтов по кварталам (дата начала дефолта находится в диапазоне 2023-2025)")

plt.tight_layout()
plt.show()



### 8.2. Среднее время прохождения дефолта (все исторические данные)

Для компаний с закрытым дефолтом (`end_date` определена) рассчитываем длительность = `end_date − start_date` (в днях).
Используется **вся** выгрузка дефолтов (`df_def`), без фильтра по периоду.

In [ ]:
# Длительность закрытых дефолтов (вся выгрузка)
df_closed = df_def[df_def["end_date"].notna()].copy()
df_closed["duration_days"] = (df_closed["end_date"] - df_closed["start_date"]).dt.days
df_closed = df_closed[df_closed["duration_days"] >= 0]

dur = df_closed["duration_days"]

max_days = int(dur.quantile(0.95)) + 50
bins = range(0, max_days, 30)

fig, ax = plt.subplots(figsize=(14, 5))
ax.hist(dur.clip(upper=max_days), bins=bins, color="steelblue", edgecolor="white")
ax.axvline(dur.mean(), color="tomato", ls="--", lw=2, label=f"Среднее ({dur.mean():.0f} дн.)")
ax.axvline(dur.median(), color="orange", ls="--", lw=2, label=f"Медиана ({dur.median():.0f} дн.)")
ax.set_xlabel("Длительность дефолта (дни)")
ax.set_ylabel("Количество компаний")
ax.set_title("Распределение длительности закрытых дефолтов (все данные)")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ["Среднее", "Медиана"],
    [dur.mean(), dur.median()],
    color=["tomato", "steelblue"], edgecolor="white"
)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{bar.get_height():.0f}", ha="center", fontsize=11)
ax.set_ylabel("Дни")
ax.set_title("Среднее и медианное время в дефолте")
plt.tight_layout()
plt.show()

### 8.3. Среднее время в дефолте по тяжести дефолта

Сравниваем длительность закрытых дефолтов в зависимости от тяжести (`severity`): тяжёлый, надзор, вышел из дефолта.

In [ ]:
sev_agg = (
    df_closed.groupby("severity")["duration_days"]
    .agg(["count", "mean", "median"])
    .round(0)
    .rename(columns={"count": "Записей", "mean": "Среднее (дни)", "median": "Медиана (дни)"})
    .sort_values("Среднее (дни)", ascending=False)
)

display(sev_agg)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(sev_agg))
w = 0.35
bars1 = ax.bar(x - w/2, sev_agg["Среднее (дни)"], w, label="Среднее",  color="tomato",    edgecolor="white")
bars2 = ax.bar(x + w/2, sev_agg["Медиана (дни)"], w, label="Медиана", color="steelblue", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(sev_agg.index, rotation=0)
ax.set_ylabel("Дни")
ax.set_title("Средняя и медианная длительность дефолта по тяжести")
ax.legend()
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{bar.get_height():.0f}", ha="center", fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{bar.get_height():.0f}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## 9. Тепловые карты корреляций ФП по сегментам (ККБ, МСБ, МкБ)

Строим отдельные тепловые карты корреляций ФП для трёх бизнес-сегментов:
1. **МкБ** — микробизнес
2. **МСБ** — малый и средний бизнес
3. **ККБ** — крупный корпоративный бизнес

In [ ]:
TOP_N = 30

for seg_name in SEG_ORDER:
    subset = dataset[dataset["segment"] == seg_name]
    n_companies = len(subset)
    n_defaults = int(subset["default_flag"].sum()) if n_companies > 0 else 0

    if n_companies > 0:
    else:

    if n_companies < 10:
        continue

    fp_freq_seg = subset[fp_cols].sum().sort_values(ascending=False)
    active_fp = fp_freq_seg[fp_freq_seg >= 2].head(TOP_N).index.tolist()

    if len(active_fp) < 2:
        continue

    corr_seg = subset[active_fp].corr()
    mask_tri = np.triu(np.ones(corr_seg.shape, dtype=bool), k=1)

    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        corr_seg, mask=mask_tri, annot=False, cmap="RdBu_r", center=0,
        vmin=-0.5, vmax=0.5, linewidths=0.3,
        xticklabels=True, yticklabels=True, ax=ax,
    )
    ax.set_title(f"Корреляции между ТОП-{len(active_fp)} ФП — {seg_name}\n"
                 f"(компаний: {n_companies:,}, дефолтных: {n_defaults:,})")
    plt.xticks(fontsize=7, rotation=90)
    plt.yticks(fontsize=7)
    plt.tight_layout()
    plt.show()

    pairs_seg = (
        corr_seg.where(np.triu(np.ones(corr_seg.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    pairs_seg.columns = ["ФП_1", "ФП_2", "Корреляция"]
    strong_seg = pairs_seg[pairs_seg["Корреляция"].abs() > 0.3].sort_values(
        "Корреляция", key=abs, ascending=False
    )

    if len(strong_seg) > 0:
        display(strong_seg.head(10).style.hide(axis="index").format({"Корреляция": "{:.3f}"}))
    else:


## 10. Кросс-таблицы «ФП × Дефолт» по сегментам

In [ ]:
cross_by_seg = {}

for seg in SEG_ORDER:
    seg_data = dataset[dataset["segment"] == seg]
    if len(seg_data) == 0:
        continue

    seg_base_rate = seg_data["default_flag"].mean()
    seg_total = len(seg_data)
    seg_results = []

    for col in fp_cols:
        n_with = (seg_data[col] == 1).sum()
        if n_with == 0:
            continue
        n_def_with = seg_data.loc[seg_data[col] == 1, "default_flag"].sum()
        rate = n_def_with / n_with if n_with > 0 else 0
        lift = rate / seg_base_rate if seg_base_rate > 0 else 0
        seg_results.append({
            "Тип ФП": col,
            "Компаний с ФП": n_with,
            "% от всех": round(n_with / seg_total * 100, 2),
            "Дефолтов": n_def_with,
            "Доля дефолтов": round(rate * 100, 2),
            "Lift": round(lift, 2),
        })

    df_seg_cross = pd.DataFrame(seg_results).sort_values("Lift", ascending=False)
    cross_by_seg[seg] = df_seg_cross

    top = df_seg_cross[df_seg_cross["Дефолтов"] >= 3].head(20)
    if len(top) > 0:
        display(
            top.style.hide(axis="index")
            .format({"Доля дефолтов": "{:.2f}%", "% от всех": "{:.2f}%"})
            .bar(subset=["Lift"], color="salmon")
        )
    else:


## 11. Топ комбинаций ФП по сегментам

In [ ]:
combos_by_seg = {}

for seg in SEG_ORDER:
    seg_data = dataset[dataset["segment"] == seg]
    if len(seg_data) < 30:
        continue
    seg_base_rate = seg_data["default_flag"].mean()
    if seg_base_rate == 0:
        continue

    seg_fp = [c for c in fp_cols if (seg_data[c] == 1).sum() >= 10]
    y_seg = seg_data["default_flag"].values
    X_seg = seg_data[seg_fp].values
    fp_n = np.array(seg_fp)

    pair_res = []
    for i, j in combinations(range(len(seg_fp)), 2):
        m = (X_seg[:, i] == 1) & (X_seg[:, j] == 1)
        n = m.sum()
        if n < 5:
            continue
        nd = y_seg[m].sum()
        if nd < 2:
            continue
        r = nd / n
        pair_res.append({
            "Комбинация": f"{fp_n[i]}  +  {fp_n[j]}",
            "Размер": 2, "Компаний": n, "Дефолтов": nd,
            "Default rate": round(r * 100, 2),
            "Lift": round(r / seg_base_rate, 2),
        })

    df_p = pd.DataFrame(pair_res).sort_values("Lift", ascending=False) if pair_res else pd.DataFrame()

    top_lift = (
        cross_by_seg.get(seg, pd.DataFrame())
        .query("`Компаний с ФП` >= 10")
        .sort_values("Lift", ascending=False)
        .head(20)["Тип ФП"].tolist()
    )
    ti = [k for k, name in enumerate(seg_fp) if name in top_lift]

    triple_res = []
    for i, j, k in combinations(ti, 3):
        m = (X_seg[:, i] == 1) & (X_seg[:, j] == 1) & (X_seg[:, k] == 1)
        n = m.sum()
        if n < 5:
            continue
        nd = y_seg[m].sum()
        if nd < 2:
            continue
        r = nd / n
        triple_res.append({
            "Комбинация": f"{fp_n[i]}  +  {fp_n[j]}  +  {fp_n[k]}",
            "Размер": 3, "Компаний": n, "Дефолтов": nd,
            "Default rate": round(r * 100, 2),
            "Lift": round(r / seg_base_rate, 2),
        })

    df_t = pd.DataFrame(triple_res).sort_values("Lift", ascending=False) if triple_res else pd.DataFrame()
    df_combo = pd.concat([df_p, df_t], ignore_index=True).sort_values("Lift", ascending=False)
    combos_by_seg[seg] = df_combo

    if len(df_combo) > 0:
        display(
            df_combo.head(10)
            .style.hide(axis="index")
            .format({"Default rate": "{:.2f}%"})
            .bar(subset=["Lift"], color="salmon")
        )
    else:


## 12. Сохранение датасета и результатов EDA

## Сводка по датасету


In [ ]:
print("=" * 70)
print("  СВОДКА ПО ДАТАСЕТУ")
print("=" * 70)

n_total = len(dataset)
n_def = int(dataset["default_flag"].sum())
dr = dataset["default_flag"].mean()

print(f"Период анализа:        {date_from:%d.%m.%Y} — {date_to:%d.%m.%Y}")
print(f"Уникальных компаний:   {n_total:,}")
print(f"Дефолтных:             {n_def:,}  ({dr:.2%})")
print(f"ФП-колонок:            {len(fp_cols)}")
print(f"")

print("По сегментам:")
for seg in SEG_ORDER:
    sub = dataset[dataset['segment'] == seg]
    nd = int(sub['default_flag'].sum())
    print(f"  {seg}: {len(sub):,} компаний, {nd:,} дефолтных ({sub['default_flag'].mean():.2%} DR)")

print(f"")
print("Среднее число ФП на компанию:")
fp_cnt = dataset[fp_cols].sum(axis=1)
d_cnt = fp_cnt[dataset['default_flag'] == 1]
nd_cnt = fp_cnt[dataset['default_flag'] == 0]
print(f"  Дефолтные:    {d_cnt.mean():.1f} (медиана {d_cnt.median():.0f})")
print(f"  Недефолтные:  {nd_cnt.mean():.1f} (медиана {nd_cnt.median():.0f})")
print("=" * 70)


In [ ]:
dataset.to_pickle(f"{DATA_DIR}/dataset_fp_default.pkl")

excel_path = f"{DATA_DIR}/приложение_build_dataset.xlsx"

def _trunc(name, limit=31):
    return name[:limit]

with pd.ExcelWriter(excel_path, engine="openpyxl") as w:
    dataset.to_excel(w, sheet_name="Датасет", index=False)

    for seg, df_s in cross_by_seg.items():
        df_s.to_excel(w, sheet_name=_trunc(f"Кросс-таблица ({seg})"), index=False)

    for seg, df_c in combos_by_seg.items():
        if len(df_c) > 0:
            df_c.to_excel(w, sheet_name=_trunc(f"Комбинации ({seg})"), index=False)



    pd.DataFrame(seg_stats).to_excel(w, sheet_name="Статистика ФП по сегм.", index=False)

print(f"Сохранено: {excel_path}")